In [19]:

from langchain_community.document_loaders import WebBaseLoader
from dotenv import load_dotenv
from bs4 import BeautifulSoup

from dotenv import load_dotenv
import os 
from langchain_groq import ChatGroq

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import CohereEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.agents import create_agent
from langchain_community.tools import tool
from langchain_cohere import CohereEmbeddings

load_dotenv()

True

In [20]:
url = "https://python.langchain.com/docs/..."

loader=WebBaseLoader(url)

docs=loader.load()

print(len(docs))

print(docs)

print(print(docs[0].page_content[:2000]))

1
[Document(metadata={'source': 'https://python.langchain.com/docs/...', 'title': 'LangChain overview - Docs by LangChain', 'description': 'LangChain provides create_agent: a minimal, highly configurable agent harness. Compose exactly the agent your use case needs from model, tools, prompt, and middleware.', 'language': 'en'}, page_content='LangChain overview - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what\'s next for agents. Get your tickets →Docs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangChain overviewOverviewDeep AgentsManaged Deep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonOverviewGet startedInstallQuickstartChangelogPhilosophyCore componentsAgentsModelsMessagesToo

In [21]:
groq_key=os.getenv("GROQ_API_KEY")
cohere_key=os.getenv("COHERE_API_KEY")
print("Environment Varibale loaded")

Environment Varibale loaded


In [22]:


splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)

chunks = splitter.split_documents(docs)

print("Documents:", len(docs))
print("Chunks:", len(chunks))

Documents: 1
Chunks: 14


In [23]:
print(chunks[2].page_content)
print(chunks[3].page_content)

LangChain vs. LangGraph vs. Deep AgentsStart with Deep Agents for a “batteries-included” agent with features like automatic context compression, a virtual filesystem, and subagent-spawning. Deep Agents are built on LangChain agents which you can also use directly.Use LangChain (create_agent) for a highly customizable harness, easily tailored to your use case and data.Use LangGraph, our low-level orchestration framework, for advanced needs combining deterministic and agentic workflows.Use LangSmith to trace, debug, and evaluate agents built with any of these frameworks. Follow the tracing quickstart to get set up. We recommend you also set up LangSmith Engine which monitors your traces, detects issues, and proposes fixes.
​ Create an agent
This example demonstrates how to create a simple LangChain agent with a custom tool:
OpenAIGoogle GeminiClaude (Anthropic)OpenRouterFireworksBasetenOllamaAzureAWS BedrockHuggingFace# pip install -qU langchain "langchain[openai]"
OpenAIGoogle GeminiCla

In [24]:


cohere_api_key = os.getenv("COHERE_API_KEY")

embedding_model = CohereEmbeddings(
    model="embed-english-light-v3.0",
    cohere_api_key=cohere_api_key)

In [26]:
vector_store=FAISS.from_documents(chunks,embedding_model)
print("CHUNKS ARE STORED IN VECTOR DB",vector_store.index.ntotal)


CHUNKS ARE STORED IN VECTOR DB 14


In [27]:
test_query="what is langchain"

top_matches=vector_store.similarity_search(test_query,k=2)
print(f"Query:{test_query}\n")
for i, match in enumerate(top_matches,start=1):
    print(match.page_content)
    print()

Query:what is langchain

See the Installation instructions and Quickstart guide to get started building your own agents and applications with LangChain.
Use LangSmith to trace requests, debug agent behavior, and evaluate outputs. Set LANGSMITH_TRACING=true and your API key to get started.
​ Core benefits

LangChain vs. LangGraph vs. Deep AgentsStart with Deep Agents for a “batteries-included” agent with features like automatic context compression, a virtual filesystem, and subagent-spawning. Deep Agents are built on LangChain agents which you can also use directly.Use LangChain (create_agent) for a highly customizable harness, easily tailored to your use case and data.Use LangGraph, our low-level orchestration framework, for advanced needs combining deterministic and agentic workflows.Use LangSmith to trace, debug, and evaluate agents built with any of these frameworks. Follow the tracing quickstart to get set up. We recommend you also set up LangSmith Engine which monitors your traces

In [28]:
#TOOL
retriever=vector_store.as_retriever(search_kwargs={"k":3})
@tool
def search_documentation(question:str)->str:
    """Search the langchain documentation for the latest information  """
    matching_chunks=retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks)


In [29]:
llm=ChatGroq(model="openai/gpt-oss-20b",temperature=0)

llm.model_name

'openai/gpt-oss-20b'

In [32]:
documentation_assistant = create_agent(
    model=llm,
    tools=[search_documentation],
    system_prompt=(
        "You are a helpful LangChain documentation assistant. "
        "Always use the search_documentation tool to look up relevant "
        "LangChain documentation before answering questions. "
        "Answer based only on the information returned by the tool. "
        "If the answer cannot be found in the documentation, "
        "say you don't know instead of guessing."
    )
)

print("LangChain documentation assistant is ready to answer questions.")

LangChain documentation assistant is ready to answer questions.


In [34]:
def ask_documentation_assistant(question:str) -> str:
    """Send  a question to the rag agent and print a nicely formatted answer."""
    print("="*60)
    print("QUESTION:",question)
    print("-"*60)
    response=documentation_assistant.invoke({"messages":[{"role":"user","content":question}]})
    answer=response["messages"][-1].content

    print("ANsWer",answer)
    print("="*60)

ask_documentation_assistant("explain langchain")

QUESTION: explain langchain
------------------------------------------------------------
ANsWer **LangChain** is a framework that helps you build applications powered by large language models (LLMs). It provides a set of core components and patterns that let you:

| Core Concept | What it does |
|--------------|--------------|
| **Agents** | Orchestrate LLM calls, decide which tool to use, and manage the overall workflow. |
| **Models** | Wrap any LLM provider (OpenAI, Gemini, Claude, etc.) so you can call it consistently. |
| **Messages** | Represent the conversation history that the LLM sees. |
| **Tools** | External functions (e.g., web search, database queries) that an agent can invoke. |
| **Memory** | Short‑term or long‑term storage of context for the agent. |
| **Middleware** | Intercept and modify requests/responses (e.g., logging, retry logic). |

### How it fits into the ecosystem

| Option | When to use it |
|--------|----------------|
| **Deep Agents** | “Batteries‑included